# Hermes 3: interactive Colab smoke test

This is a **model** trial for NousResearch/Hermes-3-Llama-3.1-8B, not the separate Hermes Agent software. It generates one reviewable text draft from a synthetic task. It does not run tools, execute generated code, run a background worker, expose a network service, or connect Muse.

1. In Colab select **Runtime → Change runtime type → T4 GPU** (or another NVIDIA GPU with at least 12 GiB free).
2. Run the cells in order. The model download is large (roughly 16 GB of weights before local 4-bit quantization) and consumes session storage and bandwidth. Runtime availability and model access are not guaranteed.
3. The final cell downloads a JSON receipt. Keep it out of public GitHub if it includes private task data. Edit the task only with information you are comfortable sending to the Colab runtime.
4. After this succeeds, the separate integration step is to install/configure Hermes Agent with a supported inference endpoint and verify tool calling. This notebook does not establish that connection.

This is an interactive notebook job; do not use it as a persistent remote-control server or distributed worker. Save results before the session ends.

In [ ]:
import os, shutil, sys
import torch

print("Python:", sys.version.split()[0])
print("Torch:", torch.__version__)
print("Free disk GiB:", round(shutil.disk_usage("/content").free / 2**30, 1))
if not torch.cuda.is_available():
    raise RuntimeError("No NVIDIA GPU detected. Select a GPU runtime before installing or downloading.")
free_bytes, total_bytes = torch.cuda.mem_get_info()
print("GPU:", torch.cuda.get_device_name(0))
print("Free / total GPU GiB:", round(free_bytes / 2**30, 1), "/", round(total_bytes / 2**30, 1))
if free_bytes < 12 * 2**30:
    raise RuntimeError("Less than 12 GiB GPU memory is free; stop rather than risk an oversized load.")
if shutil.disk_usage("/content").free < 22 * 2**30:
    raise RuntimeError("Less than 22 GiB disk is free; model download may not fit.")
print("Preflight passed. Download occurs only in a later cell.")

In [ ]:
import subprocess
# Packages only; use Colab's existing PyTorch. Version pins are a reproducible starting point.
subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                "transformers==4.48.3", "accelerate==1.3.0",
                "bitsandbytes==0.45.2", "huggingface_hub==0.28.1"],
               check=True)
print("Dependencies installed.")

In [ ]:
from huggingface_hub import HfApi
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
import transformers, accelerate, bitsandbytes

MODEL_ID = "NousResearch/Hermes-3-Llama-3.1-8B"
# Resolve once and use the same immutable model revision for tokenizer and weights.
MODEL_REVISION = HfApi().model_info(MODEL_ID).sha
print("Model revision:", MODEL_REVISION)
quant = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4",
                           bnb_4bit_compute_dtype=torch.float16)
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, revision=MODEL_REVISION,
                                           trust_remote_code=False)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID, revision=MODEL_REVISION, quantization_config=quant,
    torch_dtype=torch.float16, device_map="auto", trust_remote_code=False)
model.eval()
print("Model loaded. No agent tools have been enabled.")

In [ ]:
import datetime, hashlib, json, platform
TASK = """Given a toy actor-tracking experiment, propose one short synthetic test
that could distinguish tracking which actor last set a value from relying on the
final operation's name. Include the inputs, expected result, and one limitation.
Do not claim a real experiment was run."""
messages = [
    {"role": "system", "content": "You are a research assistant. Label proposals as untested and do not invent results."},
    {"role": "user", "content": TASK},
]
inputs = tokenizer.apply_chat_template(messages, tokenize=True, add_generation_prompt=True,
                                        return_tensors="pt", return_dict=True).to(model.device)
with torch.inference_mode():
    output = model.generate(**inputs, max_new_tokens=256, do_sample=False,
                            pad_token_id=tokenizer.eos_token_id)
answer = tokenizer.decode(output[0][inputs["input_ids"].shape[-1]:], skip_special_tokens=True)
receipt = {
    "created_utc": datetime.datetime.now(datetime.timezone.utc).isoformat(),
    "kind": "interactive_model_smoke_test_not_agent_execution",
    "model": MODEL_ID, "model_revision": MODEL_REVISION,
    "gpu": torch.cuda.get_device_name(0), "torch": torch.__version__,
    "transformers": transformers.__version__, "accelerate": accelerate.__version__,
    "bitsandbytes": bitsandbytes.__version__, "quantization": "4bit-nf4-fp16-compute",
    "generation": {"max_new_tokens": 256, "do_sample": False},
    "messages": messages, "response": answer,
    "response_sha256": hashlib.sha256(answer.encode("utf-8")).hexdigest(),
    "limits": "Unreviewed model draft; no tools, independent validation, or completed experiment."
}
receipt_path = "/content/hermes_smoke_receipt.json"
with open(receipt_path, "w", encoding="utf-8") as f:
    json.dump(receipt, f, indent=2, ensure_ascii=False)
print(answer)
print("\nReceipt saved:", receipt_path)

In [ ]:
from google.colab import files
files.download(receipt_path)